# NemoNexus - OLYMPUS Ensemble for ARC Prize 2025

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from typing import Dict, List, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
else:
    print('No GPU available - using CPU')

In [ ]:
# Clone repository to get the actual OLYMPUS ensemble architecture
!git clone https://github.com/AutomataControls/AutomataNexus_Olympus_AGI2.git /tmp/olympus
import sys
sys.path.append('/tmp/olympus')

# Import the actual OLYMPUS ensemble that matches the trained weights
from src.models.olympus_ensemble import OlympusEnsemble

In [ ]:
# Remove the simplified classes - we're using the real OLYMPUS ensemble

In [ ]:
# Use the actual OLYMPUS ensemble that matches the trained weights
class NemoNexusOLYMPUS:
    def __init__(self, model_path):
        # Force CPU device to match model weights
        self.device = torch.device('cpu')
        self.olympus = OlympusEnsemble(max_grid_size=30, d_model=256, device='cpu')
        
        # Load the trained ensemble weights
        if os.path.exists(model_path):
            try:
                checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
                if 'ensemble_state_dict' in checkpoint:
                    self.olympus.load_state_dict(checkpoint['ensemble_state_dict'], strict=False)
                    print(f'Loaded NemoNexus ensemble_state_dict from {model_path}')
                else:
                    self.olympus.load_state_dict(checkpoint, strict=False) 
                    print(f'Loaded NemoNexus state_dict from {model_path}')
            except Exception as e:
                print(f'Error loading weights: {e}')
        else:
            print(f'Model file not found: {model_path}')
    
    def predict_single(self, input_grid, target_shape=None):
        """Predict single grid using OLYMPUS ensemble"""
        # Convert to float and use CPU (no CUDA)
        input_array = np.array(input_grid, dtype=np.float32)
        
        # Create one-hot encoding (what OLYMPUS expects)
        H, W = input_array.shape
        input_onehot = np.zeros((10, H, W), dtype=np.float32)
        for i in range(10):
            input_onehot[i] = (input_array == i).astype(np.float32)
        
        # Force CPU tensor (no .to(device))
        input_tensor = torch.tensor(input_onehot, dtype=torch.float32).unsqueeze(0)
        
        try:
            with torch.no_grad():
                decision = self.olympus(input_tensor, mode='inference')
                output_logits = decision.prediction
                output_pred = torch.argmax(output_logits, dim=1)
                pred_grid = output_pred[0].cpu().numpy()
        except Exception as e:
            print(f'Prediction error: {e}')
            # Fallback to input grid if prediction fails
            pred_grid = input_array.astype(int)
        
        return pred_grid

In [ ]:
class NemoNexus:
    def __init__(self, model_dir=None):
        model_path = os.path.join(model_dir, 'NemoNexus.pt') if model_dir else None
        self.olympus_system = NemoNexusOLYMPUS(model_path)
        
    def predict(self, task_data):
        predictions = []
        test_inputs = task_data['test']
        
        for test_input in test_inputs:
            input_grid = np.array(test_input['input'])
            pred_grid = self.olympus_system.predict_single(input_grid)
            
            predictions.append({
                "attempt_1": pred_grid.tolist(),
                "attempt_2": pred_grid.tolist()
            })
        
        return predictions

In [ ]:
nemo = NemoNexus(model_dir='/kaggle/input/nemonexus/pytorch/default/1')
print('NemoNexus OLYMPUS ensemble initialized')

In [ ]:
with open('/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json', 'r') as f:
    test_challenges = json.load(f)

print(f'Loaded {len(test_challenges)} test challenges')

In [ ]:
solutions = {}
processed = 0

for task_id, task_data in test_challenges.items():
    try:
        predictions = nemo.predict(task_data)
        solutions[task_id] = predictions
        processed += 1
        
        if processed % 50 == 0:
            print(f'NemoNexus processed {processed}/{len(test_challenges)} tasks')
            
    except Exception as e:
        print(f'Error on task {task_id}: {e}')
        solutions[task_id] = [{"attempt_1": [[0]], "attempt_2": [[0]]}]

print(f'NemoNexus generated solutions for {len(solutions)} tasks')

In [ ]:
try:
    with open('submission.json', 'w') as f:
        json.dump(solutions, f, separators=(',', ':'))
    print('NemoNexus submission saved!')
    print(f'Total tasks: {len(solutions)}')
    print('Submission file: submission.json')
    
except Exception as e:
    print(f'Error saving submission: {e}')